In [25]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

This notebook creates the growth allocation list from the local growth assignments. The path to the log is here: "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log.xlsx" this should be updated anytime new growth is reported and used as a reference and read in here. Images and lat/lons are here: "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log_Residential.docx", "G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log_NonResidential.docx".    

Process: took the input from our locals (name and date recorded in spreadsheet) and found them on the map. Drew and then digitized a polygon and GIS analyst returns a list of Census blocks with the percent they are covered by these polygons. In this notebook we'll take the total growth reported and allocate it by percent to these blocks.

In [26]:
df = pd.read_excel(r"G:\3-GitHub\FCLUCS_Scenario_Planning\5-documentation\New Development Log.xlsx", sheet_name="Non-Residential", usecols=["ID", "Number of Jobs Modeled"])

In [27]:
df

,ID,Number of Jobs Modeled
0,NR1,150
1,NR2,25
2,NR3,25
3,NR4,45
4,NR5,35
5,NR6,35
6,NR7,20


In [28]:
df_perc = pd.read_excel(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\2-interim/StudyArea_Censusblocksummary.xls")

So this is filtered down where if a block is less than 50% intersected with the polygon, we don't allocate anything to it (then normalize percentages so it still allocates 100% of the value). Logic being that these values are taken as truth and we don't want to disrupt what's appearing there for something with potentially lower certainty/a lower value. Also it's an art not a science. However, in the case that none of the values exceed 50%... if one has 2 blocks that meet 50% threshold those two are retained, if one has 3 or more and none meet the threshold then all are retained.

In [29]:
# Identify NR developments (no intersection threshold)
df_perc_filtered = df_perc.loc[
    df_perc['ID'].str.contains('NR', na=False)
].copy()

# Re-normalize percentages to sum to 100% per development
df_perc_filtered['PCT_INTERSECT_NORM'] = (
    df_perc_filtered['PCT_INTERSECT']
    / df_perc_filtered.groupby('ID')['PCT_INTERSECT'].transform('sum')
    * 100
)

# Match employment totals from development dataframe
df_perc_filtered['EMP_TOTAL'] = df_perc_filtered['ID'].map(
    df.set_index('ID')['Number of Jobs Modeled']
)

# Allocate employment proportionally to each block
df_perc_filtered['EMP_PER_BLOCK'] = (
    df_perc_filtered['EMP_TOTAL']
    * (df_perc_filtered['PCT_INTERSECT_NORM'] / 100)
)

# Match employment totals
df_perc_filtered['EMP_TOTAL'] = df_perc_filtered['ID'].map(
    df.set_index('ID')['Number of Jobs Modeled']
)

# Allocate employment
df_perc_filtered['EMP_PER_BLOCK'] = (
    df_perc_filtered['EMP_TOTAL']
    * (df_perc_filtered['PCT_INTERSECT_NORM'] / 100)
)

In [30]:
df_perc_filtered.sort_values(by = 'PCT_BLOCK', ascending = True).head(10)

,OBJECTID,ID,GEOID,PCT_INTERSECT,PCT_BLOCK,SIZE_BLOCK,PCT_INTERSECT_NORM,EMP_TOTAL,EMP_PER_BLOCK
25,26,NR6,212219702022007,0.001570,0.018589,509.234695,0.003636,35,0.001273
0,1,NR1,210472013011038,0.173787,0.502024,97.549293,0.778959,150,1.168439
1,2,NR1,210472013011036,22.136406,99.497971,151.783406,99.221041,150,148.831561
24,25,NR6,212219702022008,43.172145,99.981416,99.588439,99.996364,35,34.998727
21,22,NR5,210472013041026,1.204076,100.000000,117.810120,100.000000,35,35.000000
2,3,NR4,210472014002075,4.853372,100.000000,95.688894,100.000000,45,45.000000
23,24,NR7,212219702011014,12.909711,100.000000,147.538474,100.000000,20,20.000000
22,23,NR2,210472013041026,0.118433,100.000000,117.810120,100.000000,25,25.000000


In [31]:
df_perc_filtered = df_perc_filtered.rename(columns = {'GEOID': 'GEO_ID'})
df_perc_filtered['GEO_ID'] = df_perc_filtered['GEO_ID'].astype(str)

In [32]:
df.head()

,ID,Number of Jobs Modeled
0,NR1,150
1,NR2,25
2,NR3,25
3,NR4,45
4,NR5,35


In [33]:
allocation_check = df_perc_filtered.groupby('ID').agg(
    PCT_TOTAL=('PCT_INTERSECT_NORM', 'sum'),
    EMP_TOTAL=('EMP_TOTAL', 'first'),
    EMP_ALLOCATED=('EMP_PER_BLOCK', 'sum')
)

allocation_check['EMP_DIFF'] = (
    allocation_check['EMP_TOTAL'] - allocation_check['EMP_ALLOCATED']
)

print(allocation_check)

     PCT_TOTAL  EMP_TOTAL  EMP_ALLOCATED  EMP_DIFF
ID                                                
NR1      100.0        150          150.0       0.0
NR2      100.0         25           25.0       0.0
NR4      100.0         45           45.0       0.0
NR5      100.0         35           35.0       0.0
NR6      100.0         35           35.0       0.0
NR7      100.0         20           20.0       0.0


In [34]:
#check for missing ID's - Hannah hadn't run this one yet
missing_ids = df.loc[
    ~df['ID'].isin(df_perc_filtered['ID'])
]

print(missing_ids)

    ID  Number of Jobs Modeled
2  NR3                      25


In [35]:
#sum by GEOID and we'll output to read into the allocation function for future

#agg by census block
local_growth = (df_perc_filtered.groupby('GEO_ID')['EMP_PER_BLOCK'].sum().to_dict())

#export to JSON
output_path = Path(r"G:\3-GitHub\FCLUCS_Scenario_Planning\1-data\2-interim\local_growth_employment.json")

with open(output_path, 'w') as f:
    json.dump(local_growth, f, indent=4)